## Импорт библиотек и загрузка данных

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import numpy as np

from src.computational_functions import calc_sleep_hours

In [3]:
primary_df = pd.read_csv("../data/processed/primary_dream.csv")
primary_df.head(5)

,nick,age,sex,agreement,date,fall_asleep,wake_up,sleep_quality,awakenings,dream_remembered,...,amount_of_coffee,amount_of_coffee_after16,alcohol,strong_emotions,stress_level,hours_of_physical_activity,daytime_sleep,food_before_bed,screen_time,screen_before_bed
0,OLKA75,45 лет и более,Женский,Да,13.09.2026,23:30:00,6:45:00,3,3,Да,...,1,0,Да,Нет,4,2,Да,Нет,3,Да
1,Zverskiyponos67,до 18 лет,Женский,Да,13.09.2026,2:00:00,10:00:00,3,1,Нет,...,1,0,Нет,Нет,5,0,Нет,Нет,8,Нет
2,Kreeper,от 18 до 30 лет,Женский,Да,13.09.2026,0:00:00,8:13:00,4,0,Нет,...,0,0,Нет,Нет,2,4,Нет,Нет,8,Да
3,Tukiceja,45 лет и более,Мужской,Да,13.09.2026,23:00:00,6:20:00,3,3,Нет,...,1,0,Да,Нет,2,3,Да,Нет,3,Да
4,TihiyOmut,от 18 до 30 лет,Мужской,Да,13.09.2026,3:00:00,10:25:00,5,0,Да,...,10,4,Да,Да,2,1,Нет,Да,3,Да


## Предварительная обработка данных

In [4]:
df = primary_df.copy()

### Преобразование даты и времени

In [5]:
# Дата
df["date"] = pd.to_datetime(df["date"])

# Время
df["wake_up"] = pd.to_timedelta(df["wake_up"])
df["fall_asleep"] = pd.to_timedelta(df["fall_asleep"])

C:\Users\Ilay\AppData\Local\Temp\ipykernel_13396\2717975209.py:2: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"])


### Преобразование категориальных признаков

In [6]:
# Категориальные колонки
df["dream_colored"] = df["dream_colored"].map({"В основном цветное": "color", "В основном чёрно-белое": "black-white", "Точно не могу сказать": "not_defined"})
categorial_cols = ["dream_colored"]
df[categorial_cols] = df[categorial_cols].astype("category")

age_ordered = ["до 18 лет", "от 18 до 30 лет", "от 30 до 45  лет", "45 лет и более"]
df["age"] = pd.Categorical(df["age"], categories=age_ordered, ordered=True)

Все категориальные значения были предварительно ограничены вариантами Google Forms. На этапе контроля проверено, что после импорта присутствуют только ожидаемые категории.

### Преобразование бинарных признаков

In [7]:
# Кодировка и преобразование в числовые
df["sex"] = df["sex"].map({"Мужской": 0, "Женский": 1})

binar_dict = {"Нет": 0, "Да": 1}
binary_cols = ["dream_remembered", "alcohol", "daytime_sleep", "food_before_bed", "strong_emotions", "screen_before_bed"]
for col in binary_cols:
    df[col] = df[col].map(binar_dict)
    
for col in df.columns:
    if "int" in str(df[col].dtype):
        df[col] = pd.to_numeric(df[col], downcast="integer")

|Признак|Кодировка|Значение|
|-------|---------|--------|
|sex|0|Мужской пол|
|sex|1|Женский пол|
|dream_remembered, alcohol, daytime_sleep, food_before_bed, strong_emotions, screen_before_bed|0|Нет|
|dream_remembered, alcohol, daytime_sleep, food_before_bed, strong_emotions, screen_before_bed|1|Да|

### Расчёт продолжительности сна

In [8]:
df = calc_sleep_hours(df)
df[["sleep_duration", "sleep_hours"]]

,sleep_duration,sleep_hours
0,0 days 07:15:00,7.250000
1,0 days 08:00:00,8.000000
2,0 days 08:13:00,8.216667
3,0 days 07:20:00,7.333333
4,0 days 07:25:00,7.416667
...,...,...
415,0 days 07:40:00,7.666667
416,0 days 08:20:00,8.333333
417,0 days 06:50:00,6.833333
418,0 days 08:40:00,8.666667


## Проверка качества данных

In [9]:
# info после обработки столбцов
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype          
---  ------                      --------------  -----          
 0   nick                        420 non-null    str            
 1   age                         420 non-null    category       
 2   sex                         420 non-null    int8           
 3   agreement                   420 non-null    str            
 4   date                        420 non-null    datetime64[us] 
 5   fall_asleep                 420 non-null    timedelta64[us]
 6   wake_up                     420 non-null    timedelta64[us]
 7   sleep_quality               420 non-null    int8           
 8   awakenings                  420 non-null    int8           
 9   dream_remembered            420 non-null    int8           
 10  dream_colored               187 non-null    category       
 11  amount_of_coffee            420 non-null    int8        

In [10]:
print("Пропусков после бинаризации:", df[binary_cols].isna().sum().sum())

Пропусков после бинаризации: 0


In [11]:
df.isna().sum()

nick                            0
age                             0
sex                             0
agreement                       0
date                            0
fall_asleep                     0
wake_up                         0
sleep_quality                   0
awakenings                      0
dream_remembered                0
dream_colored                 233
amount_of_coffee                0
amount_of_coffee_after16        0
alcohol                         0
strong_emotions                 0
stress_level                    0
hours_of_physical_activity      0
daytime_sleep                   0
food_before_bed                 0
screen_time                     0
screen_before_bed               0
sleep_duration                  0
sleep_hours                     0
dtype: int64

Пропуски найдены только в столбце dream_colored: 234. Данные пропуски являются не случайными, а техническими, потому что в этот день пользователи не видели сна ночью.

dream_remembered=="Нет" --> dream_colored=="NaN"

In [12]:
without_dream = df[df["dream_remembered"]== 0]["dream_remembered"].count()
print(f"Количество ответов людей, не видивших сон: {without_dream}")

Количество ответов людей, не видивших сон: 233


In [13]:
df.describe()

,sex,date,fall_asleep,wake_up,sleep_quality,awakenings,dream_remembered,amount_of_coffee,amount_of_coffee_after16,alcohol,strong_emotions,stress_level,hours_of_physical_activity,daytime_sleep,food_before_bed,screen_time,screen_before_bed,sleep_duration,sleep_hours
count,420.000000,420,420,420,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420.000000,420,420.000000
mean,0.645238,2026-09-18 15:05:08.571428,0 days 09:46:24.428571,0 days 07:51:21.285714,3.745238,1.257143,0.445238,0.790476,0.264286,0.097619,0.452381,3.633333,2.530952,0.128571,0.461905,5.533333,0.909524,0 days 07:27:14,7.453889
min,0.000000,2026-09-13 00:00:00,0 days 00:00:00,0 days 00:09:00,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0 days 01:00:00,1.000000
25%,0.000000,2026-09-16 00:00:00,0 days 01:00:00,0 days 06:30:00,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,0.000000,0.000000,3.000000,1.000000,0 days 06:20:00,6.333333
50%,1.000000,2026-09-19 00:00:00,0 days 02:47:30,0 days 07:40:00,4.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,2.000000,0.000000,0.000000,5.000000,1.000000,0 days 07:22:00,7.366667
75%,1.000000,2026-09-22 00:00:00,0 days 23:00:00,0 days 09:00:00,5.000000,2.000000,1.000000,1.000000,0.000000,0.000000,1.000000,5.000000,4.000000,0.000000,1.000000,8.000000,1.000000,0 days 08:30:00,8.500000
max,1.000000,2026-09-26 00:00:00,0 days 23:59:00,0 days 16:12:00,5.000000,10.000000,1.000000,10.000000,4.000000,1.000000,1.000000,10.000000,10.000000,1.000000,1.000000,10.000000,1.000000,0 days 23:14:00,23.233333
std,0.479012,NaN,0 days 10:23:09.495422,0 days 01:55:03.373134,1.070031,1.639481,0.497585,1.119966,0.569018,0.297153,0.498321,2.297416,1.995578,0.335124,0.499141,2.694701,0.287205,0 days 02:14:01.201669,2.233667


В ряде количественных признаков среднее значение превышает медиану, что может указывать на правостороннюю асимметрию распределения. Некоторые экстремальные значения требуют дополнительной проверки.

In [14]:
# Дубликаты
df.duplicated().sum()

np.int64(0)

На стадии проверки качества данных дупликаты не были обнаружены

In [15]:
# Уникальные значения
df.nunique()

nick                           53
age                             4
sex                             2
agreement                       1
date                           14
fall_asleep                   117
wake_up                       118
sleep_quality                   5
awakenings                     11
dream_remembered                2
dream_colored                   3
amount_of_coffee                6
amount_of_coffee_after16        4
alcohol                         2
strong_emotions                 2
stress_level                   10
hours_of_physical_activity     10
daytime_sleep                   2
food_before_bed                 2
screen_time                    11
screen_before_bed               2
sleep_duration                150
sleep_hours                   150
dtype: int64

In [16]:
# Все значения столбца "agreement" содержат только ответ "Да"
df = df.drop(["agreement"], axis=1)

Столбец agreement удалён, поскольку все участники дали одинаковый ответ и признак не содержит вариативности.

## Первичный анализ выборки

### Анализ количества участников и валидность псевдонимов

In [17]:
print(f"Количество уникальных пользователей: {df["nick"].nunique()}")

Количество уникальных пользователей: 53


In [18]:
df["nick"].unique()

<StringArray>
[          'OLKA75',  'Zverskiyponos67',          'Kreeper',
         'Tukiceja',        'TihiyOmut',         'aboba777',
      'SLEEP_LOVER',            'hewwa',            'Прага',
              'Оно',           'nejnej',         'krasotka',
            'FOX07',             'Leva',         'Удача777',
          '1515net',      'Птицадивная',       'nikita_pdf',
    'Enemyofdeath ',        'aquamana ',          'GRASS42',
          'Маркул ',       'Константин',           'Наташа',
              'son',    'Питер Паркер ',       'silverain ',
 'GangsterMarmilad',         'WH1teM4N',            'Настя',
          'Rinasvm',           'Deston',            'груша',
        'даринкинс',     'Enemyofdeath',               'N1',
           'Fit345',        'Tukiceja ',     'Питер Паркер',
         'aquamana',           'Маркул',     'Птицадивная ',
        'silverain',       'TihiyOmut ',          'Pandora',
        'WH1teM4N ',      'nikita_pdf ',       'Sunflower ',
        'S

Обнаружены ошибки с наличием пробелов в конце, начале и середине псевдонима

In [19]:
# Очистим лишние пробелы
df["nick"] = df["nick"].str.replace(" ", "")
print(f"Количество уникальных участников: {df["nick"].nunique()}")

Количество уникальных участников: 38


### Количество наблюдений на каждого участника опроса

In [20]:
# Установим для каждого пользователя уникальный participant_id
df["participant_id"] = df.groupby(by=["nick"])["nick"].transform("ngroup") + 1

df = df.drop("nick", axis=1)

In [21]:
person_day = df.groupby(by=["participant_id"]).size().reset_index(name='count')
person_day

,participant_id,count
0,1,12
1,2,4
2,3,12
3,4,12
4,5,11
5,6,12
6,7,12
7,8,12
8,9,12
9,10,7


In [22]:
print(f"Количество участников, проходивших опрос все 12 дней: {len(person_day[person_day["count"] >= 12])}")

Количество участников, проходивших опрос все 12 дней: 26


### Распределение возрастных групп

In [23]:
df[["participant_id", "age"]].drop_duplicates()["age"].value_counts(normalize=True)

age
от 18 до 30 лет     0.692308
45 лет и более      0.205128
до 18 лет           0.076923
от 30 до 45  лет    0.025641
Name: proportion, dtype: float64

Возраст 69% в группе "от 18 до 30 лет", признак мало вариативен и почти константа

### Распределение пола

In [24]:
df[["participant_id", "sex"]].drop_duplicates()["sex"].value_counts(normalize=True)

sex
1    0.657895
0    0.342105
Name: proportion, dtype: float64

Доля женского пола (65.7%), проходившего опрос, значительно больше мужского (34,2%)

### Распределение целевой переменной `dream_remembered`

In [25]:
df["dream_remembered"].value_counts(normalize=True)

dream_remembered
0    0.554762
1    0.445238
Name: proportion, dtype: float64

In [26]:
# Сохранение в csv
df.to_csv("../data/processed/cleaned_and_encoded.csv", index=False)